## CEAS-08 Dataset, Real Data vs Synthetic with Rewriting

In [ ]:
# Advanced Semantic Comparison Methods for Real vs Multiple Synthetic Datasets
# Jupyter Notebook Implementation - Multi-Dataset Comparison

import pandas as pd
import numpy as np
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Advanced semantic analysis libraries
from sentence_transformers import SentenceTransformer, util
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA, LatentDirichletAllocation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import wasserstein_distance, ks_2samp
from scipy.spatial.distance import pdist, squareform
import umap

print("Libraries imported successfully!")

# =============================================================================
# 1. SETUP AND DATA LOADING - MULTIPLE SYNTHETIC DATASETS
# =============================================================================

# File paths
TRAIN_FILE = "../raw/email_phishing_CEAS-08_train.csv.gz"
SYNTHETIC_FILES = {
    'original': "../data/rewrite/email_phishing_CEAS-08_malicious_original_rewritten.csv.gz",
    'strong': "../data/rewrite/email_phishing_CEAS-08_malicious_strong_rewritten.csv.gz",
    'weak': "../data/rewrite/email_phishing_CEAS-08_malicious_weak_rewritten.csv.gz"
}

EMBEDDING_DIR = Path("../embedding")
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

def load_and_preprocess_data():
    """Load and preprocess the datasets including multiple synthetic variants"""
    print("Loading datasets...")
    
    # Load training data and filter malicious samples
    if TRAIN_FILE.endswith('.gz'):
        train_df = pd.read_csv(TRAIN_FILE, compression='gzip')
    else:
        train_df = pd.read_csv(TRAIN_FILE)
    
    real_malicious_df = train_df[train_df['label'] == 1].copy()
    print(f"Real malicious samples: {len(real_malicious_df)}")
    
    # Load multiple synthetic datasets
    synthetic_dfs = {}
    for variant, file_path in SYNTHETIC_FILES.items():
        if file_path.endswith('.gz'):
            df = pd.read_csv(file_path, compression='gzip')
        else:
            df = pd.read_csv(file_path)
        
        synthetic_dfs[variant] = df
        print(f"Synthetic {variant} samples: {len(df)}")
    
    # Preprocess text for all datasets
    all_dfs = [real_malicious_df] + list(synthetic_dfs.values())
    for df in all_dfs:
        df['subject'] = df['subject'].fillna('')
        df['body'] = df['body'].fillna('')
        df['combined_text'] = df['subject'].astype(str) + ' ' + df['body'].astype(str)
        df['combined_text'] = df['combined_text'].str.strip()
    
    return real_malicious_df, synthetic_dfs

# Load data
real_df, synthetic_dfs = load_and_preprocess_data()
real_texts = real_df['combined_text'].tolist()

# Extract texts for each synthetic variant
synthetic_texts = {}
for variant, df in synthetic_dfs.items():
    synthetic_texts[variant] = df['combined_text'].tolist()

print(f"\nLoaded datasets:")
print(f"Real texts: {len(real_texts)}")
for variant, texts in synthetic_texts.items():
    print(f"Synthetic {variant}: {len(texts)}")

# =============================================================================
# 2. MULTI-DATASET SENTENCE TRANSFORMER EMBEDDINGS GENERATION
# =============================================================================

def generate_sentence_embeddings(texts, model_name, dataset_name):
    """Generate sentence transformer embeddings with caching"""
    embedding_file = EMBEDDING_DIR / f"{dataset_name}_{model_name.replace('/', '_')}_embeddings.pkl"
    
    if embedding_file.exists():
        print(f"Loading cached embeddings: {embedding_file}")
        with open(embedding_file, 'rb') as f:
            return pickle.load(f)
    
    print(f"Generating {model_name} embeddings for {dataset_name}...")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    
    # Save embeddings
    with open(embedding_file, 'wb') as f:
        pickle.dump(embeddings, f)
    
    print(f"Saved embeddings: {embeddings.shape}")
    return embeddings

# Generate multiple types of embeddings for all datasets
embedding_models = {
    'minilm': 'all-MiniLM-L6-v2',
    'mpnet': 'all-mpnet-base-v2',
    'roberta': 'all-roberta-large-v1'
}

embeddings_dict = {}

for model_key, model_name in embedding_models.items():
    try:
        print(f"\n{'='*60}")
        print(f"Processing {model_name}")
        print(f"{'='*60}")
        
        # Generate real embeddings
        real_embeddings = generate_sentence_embeddings(real_texts, model_name, f'real_{model_key}')
        
        # Generate embeddings for each synthetic variant
        synthetic_embeddings = {}
        for variant, texts in synthetic_texts.items():
            synthetic_embeddings[variant] = generate_sentence_embeddings(
                texts, model_name, f'synthetic_{variant}_{model_key}'
            )
        
        embeddings_dict[model_key] = {
            'real': real_embeddings,
            'synthetic': synthetic_embeddings,
            'model_name': model_name
        }
        
    except Exception as e:
        print(f"Error with {model_name}: {e}")
        continue

print(f"\nSuccessfully loaded {len(embeddings_dict)} embedding models")

# =============================================================================
# 3. MULTI-DATASET SEMANTIC TEXTUAL SIMILARITY (STS) ANALYSIS
# =============================================================================

def semantic_similarity_analysis_multi(real_embeddings, synthetic_embeddings_dict, model_name):
    """Comprehensive semantic similarity analysis for multiple synthetic datasets"""
    print(f"\n🔍 Multi-Dataset Semantic Similarity Analysis - {model_name}")
    print("-" * 60)
    
    results = {}
    cross_similarities = {}
    
    for variant, synthetic_embeddings in synthetic_embeddings_dict.items():
        print(f"\nAnalyzing {variant} variant...")
        
        # Cross-dataset similarity matrix
        cross_similarity = util.cos_sim(real_embeddings, synthetic_embeddings)
        cross_similarities[variant] = cross_similarity
        
        # Within-dataset similarity matrices
        real_self_sim = util.cos_sim(real_embeddings, real_embeddings)
        synthetic_self_sim = util.cos_sim(synthetic_embeddings, synthetic_embeddings)
        
        # Remove diagonal for self-similarity
        real_self_sim_no_diag = real_self_sim.clone()
        real_self_sim_no_diag.fill_diagonal_(0)
        synthetic_self_sim_no_diag = synthetic_self_sim.clone()
        synthetic_self_sim_no_diag.fill_diagonal_(0)
        
        # Calculate statistics
        variant_results = {
            'cross_similarity': {
                'mean': float(cross_similarity.mean()),
                'std': float(cross_similarity.std()),
                'max': float(cross_similarity.max()),
                'min': float(cross_similarity.min()),
                'median': float(cross_similarity.median())
            },
            'real_internal_similarity': {
                'mean': float(real_self_sim_no_diag.mean()),
                'std': float(real_self_sim_no_diag.std())
            },
            'synthetic_internal_similarity': {
                'mean': float(synthetic_self_sim_no_diag.mean()),
                'std': float(synthetic_self_sim_no_diag.std())
            }
        }
        
        # Diversity scores
        variant_results['diversity_scores'] = {
            'real_diversity': 1 - variant_results['real_internal_similarity']['mean'],
            'synthetic_diversity': 1 - variant_results['synthetic_internal_similarity']['mean'],
            'diversity_gap': abs(variant_results['real_internal_similarity']['mean'] - 
                               variant_results['synthetic_internal_similarity']['mean'])
        }
        
        # Best matches analysis
        max_similarities = cross_similarity.max(dim=1)[0]
        variant_results['best_matches'] = {
            'mean_best_match': float(max_similarities.mean()),
            'std_best_match': float(max_similarities.std()),
            'high_quality_matches': float((max_similarities > 0.8).sum() / len(max_similarities)),
            'very_high_quality_matches': float((max_similarities > 0.9).sum() / len(max_similarities))
        }
        
        results[variant] = variant_results
        
        # Print variant-specific results
        print(f"  Cross-similarity: {variant_results['cross_similarity']['mean']:.4f} ± {variant_results['cross_similarity']['std']:.4f}")
        print(f"  Diversity gap: {variant_results['diversity_scores']['diversity_gap']:.4f}")
        print(f"  Best match quality: {variant_results['best_matches']['mean_best_match']:.4f}")
        print(f"  High-quality matches (>0.8): {variant_results['best_matches']['high_quality_matches']:.2%}")
        print(f"  Very high-quality matches (>0.9): {variant_results['best_matches']['very_high_quality_matches']:.2%}")
    
    return results, cross_similarities

# Run STS analysis for all models and variants
sts_results = {}
similarity_matrices = {}

for model_key, embeddings in embeddings_dict.items():
    results, cross_sims = semantic_similarity_analysis_multi(
        embeddings['real'], 
        embeddings['synthetic'], 
        embeddings['model_name']
    )
    sts_results[model_key] = results
    similarity_matrices[model_key] = cross_sims

# =============================================================================
# 4. MULTI-DATASET TOPIC MODELING COMPARISON
# =============================================================================

def topic_modeling_analysis_multi(real_texts, synthetic_texts_dict, n_topics=10):
    """Topic modeling comparison for multiple synthetic datasets"""
    print(f"\n📊 Multi-Dataset Topic Modeling Analysis")
    print("-" * 60)
    
    results = {}
    
    for variant, synthetic_texts in synthetic_texts_dict.items():
        print(f"\nAnalyzing topic distribution for {variant} variant...")
        
        # Combine texts with labels
        all_texts = real_texts + synthetic_texts
        labels = ['real'] * len(real_texts) + ['synthetic'] * len(synthetic_texts)
        
        # Initialize BERTopic
        vectorizer = CountVectorizer(
            stop_words="english", 
            max_features=1000, 
            ngram_range=(1, 2),
            min_df=2
        )
        
        topic_model = BERTopic(
            vectorizer_model=vectorizer,
            n_gram_range=(1, 2),
            min_topic_size=5,
            calculate_probabilities=True,
            verbose=False
        )
        
        topics, probabilities = topic_model.fit_transform(all_texts)
        topic_info = topic_model.get_topic_info()
        
        # Analyze topic distribution by dataset
        real_topics = topics[:len(real_texts)]
        synthetic_topics = topics[len(real_texts):]
        
        # Topic distribution comparison
        real_topic_dist = pd.Series(real_topics).value_counts(normalize=True).sort_index()
        synthetic_topic_dist = pd.Series(synthetic_topics).value_counts(normalize=True).sort_index()
        
        # Align topic distributions
        all_topics = sorted(set(real_topics + synthetic_topics))
        real_dist_aligned = [real_topic_dist.get(t, 0) for t in all_topics]
        synthetic_dist_aligned = [synthetic_topic_dist.get(t, 0) for t in all_topics]
        
        # Calculate distribution similarity
        topic_wasserstein = wasserstein_distance(real_dist_aligned, synthetic_dist_aligned)
        topic_kl_div = calculate_kl_divergence(real_dist_aligned, synthetic_dist_aligned)
        
        # Calculate topic overlap (how many topics are shared)
        real_topics_set = set(real_topics)
        synthetic_topics_set = set(synthetic_topics)
        topic_overlap = len(real_topics_set.intersection(synthetic_topics_set)) / len(real_topics_set.union(synthetic_topics_set))
        
        results[variant] = {
            'n_topics': len(topic_info),
            'topic_distribution_similarity': {
                'wasserstein_distance': topic_wasserstein,
                'kl_divergence': topic_kl_div,
                'topic_overlap': topic_overlap
            },
            'topic_info': topic_info.to_dict('records'),
            'real_topic_distribution': dict(real_topic_dist),
            'synthetic_topic_distribution': dict(synthetic_topic_dist)
        }
        
        print(f"  Topics found: {len(topic_info)}")
        print(f"  Wasserstein distance: {topic_wasserstein:.4f}")
        print(f"  KL divergence: {topic_kl_div:.4f}")
        print(f"  Topic overlap: {topic_overlap:.4f}")
    
    return results

def calculate_kl_divergence(p, q, epsilon=1e-10):
    """Calculate KL divergence between two distributions"""
    p = np.array(p) + epsilon
    q = np.array(q) + epsilon
    p = p / p.sum()
    q = q / q.sum()
    return np.sum(p * np.log(p / q))

# Run topic modeling for all variants
topic_results = topic_modeling_analysis_multi(real_texts, synthetic_texts)

# =============================================================================
# 5. MULTI-DATASET SEMANTIC DIVERSITY ANALYSIS
# =============================================================================

def semantic_diversity_analysis_multi(embeddings_dict):
    """Advanced semantic diversity analysis for multiple synthetic datasets"""
    print(f"\n🌈 Multi-Dataset Semantic Diversity Analysis")
    print("-" * 60)
    
    diversity_results = {}
    
    for model_key, embeddings in embeddings_dict.items():
        print(f"\nAnalyzing {embeddings['model_name']}...")
        model_results = {}
        
        real_emb = embeddings['real']
        
        for variant, synthetic_emb in embeddings['synthetic'].items():
            print(f"  Processing {variant} variant...")
            
            # Pairwise distance analysis
            real_distances = pdist(real_emb, metric='cosine')
            synthetic_distances = pdist(synthetic_emb, metric='cosine')
            
            # Clustering analysis
            n_clusters = min(10, len(real_emb) // 5)
            
            real_kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            synthetic_kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            
            real_clusters = real_kmeans.fit_predict(real_emb)
            synthetic_clusters = synthetic_kmeans.fit_predict(synthetic_emb)
            
            real_silhouette = silhouette_score(real_emb, real_clusters)
            synthetic_silhouette = silhouette_score(synthetic_emb, synthetic_clusters)
            
            # Coverage analysis using PCA
            real_pca = PCA(n_components=min(10, real_emb.shape[1]))
            synthetic_pca = PCA(n_components=min(10, synthetic_emb.shape[1]))
            
            real_pca_proj = real_pca.fit_transform(real_emb)
            synthetic_pca_proj = synthetic_pca.fit_transform(synthetic_emb)
            
            # Volume approximation
            real_volume = np.prod(np.std(real_pca_proj, axis=0))
            synthetic_volume = np.prod(np.std(synthetic_pca_proj, axis=0))
            
            # Nearest neighbor analysis
            def nearest_neighbor_diversity(embeddings, k=5):
                distances = squareform(pdist(embeddings, metric='cosine'))
                np.fill_diagonal(distances, np.inf)
                knn_distances = np.sort(distances, axis=1)[:, :k]
                return np.mean(knn_distances)
            
            real_nn_diversity = nearest_neighbor_diversity(real_emb)
            synthetic_nn_diversity = nearest_neighbor_diversity(synthetic_emb)
            
            # Cross-dataset analysis
            cross_distances = []
            for i in range(min(100, len(real_emb))):  # Sample for efficiency
                real_sample = real_emb[i:i+1]
                distances_to_synthetic = util.cos_sim(real_sample, synthetic_emb).flatten()
                cross_distances.extend(distances_to_synthetic.cpu().numpy())
            
            mean_cross_distance = np.mean(cross_distances)
            
            model_results[variant] = {
                'pairwise_distances': {
                    'real_mean': float(np.mean(real_distances)),
                    'real_std': float(np.std(real_distances)),
                    'synthetic_mean': float(np.mean(synthetic_distances)),
                    'synthetic_std': float(np.std(synthetic_distances)),
                    'distance_similarity': 1 - abs(np.mean(real_distances) - np.mean(synthetic_distances)),
                    'cross_dataset_similarity': float(mean_cross_distance)
                },
                'clustering_quality': {
                    'real_silhouette': float(real_silhouette),
                    'synthetic_silhouette': float(synthetic_silhouette),
                    'silhouette_similarity': 1 - abs(real_silhouette - synthetic_silhouette)
                },
                'coverage_analysis': {
                    'real_volume': float(real_volume),
                    'synthetic_volume': float(synthetic_volume),
                    'volume_ratio': float(synthetic_volume / real_volume) if real_volume > 0 else 0
                },
                'nearest_neighbor_diversity': {
                    'real_diversity': float(real_nn_diversity),
                    'synthetic_diversity': float(synthetic_nn_diversity),
                    'diversity_ratio': float(synthetic_nn_diversity / real_nn_diversity) if real_nn_diversity > 0 else 0
                }
            }
            
            print(f"    Cross-dataset similarity: {model_results[variant]['pairwise_distances']['cross_dataset_similarity']:.4f}")
            print(f"    Distance similarity: {model_results[variant]['pairwise_distances']['distance_similarity']:.4f}")
            print(f"    Volume ratio: {model_results[variant]['coverage_analysis']['volume_ratio']:.4f}")
        
        diversity_results[model_key] = model_results
    
    return diversity_results

# Run diversity analysis for all variants
diversity_results = semantic_diversity_analysis_multi(embeddings_dict)

# =============================================================================
# 6. RANKING AND COMPARISON ANALYSIS
# =============================================================================

def rank_synthetic_variants():
    """Rank synthetic variants based on similarity to real data"""
    print(f"\n🏆 RANKING SYNTHETIC VARIANTS")
    print("=" * 60)
    
    variants = list(SYNTHETIC_FILES.keys())
    ranking_results = {}
    
    # Collect metrics for ranking
    for model_key in sts_results.keys():
        model_scores = {}
        
        for variant in variants:
            # Normalize scores (higher is better for similarity)
            cross_sim = sts_results[model_key][variant]['cross_similarity']['mean']
            diversity_gap = 1 - sts_results[model_key][variant]['diversity_scores']['diversity_gap']  # Invert so lower gap = higher score
            best_match = sts_results[model_key][variant]['best_matches']['mean_best_match']
            high_quality_matches = sts_results[model_key][variant]['best_matches']['high_quality_matches']
            
            # Topic modeling scores
            topic_sim = 1 - topic_results[variant]['topic_distribution_similarity']['wasserstein_distance']  # Invert distance
            topic_overlap = topic_results[variant]['topic_distribution_similarity']['topic_overlap']
            
            # Diversity scores
            distance_sim = diversity_results[model_key][variant]['pairwise_distances']['distance_similarity']
            cross_dataset_sim = diversity_results[model_key][variant]['pairwise_distances']['cross_dataset_similarity']
            volume_ratio = diversity_results[model_key][variant]['coverage_analysis']['volume_ratio']
            volume_score = 1 - abs(1 - volume_ratio)  # Closer to 1 is better
            
            # Calculate composite score
            composite_score = (
                cross_sim * 0.25 +           # Cross-similarity (25%)
                diversity_gap * 0.15 +       # Diversity gap (15%)
                best_match * 0.20 +          # Best match quality (20%)
                high_quality_matches * 0.10 + # High quality matches (10%)
                topic_sim * 0.10 +           # Topic similarity (10%)
                topic_overlap * 0.05 +       # Topic overlap (5%)
                distance_sim * 0.10 +        # Distance similarity (10%)
                volume_score * 0.05          # Volume similarity (5%)
            )
            
            model_scores[variant] = {
                'composite_score': composite_score,
                'cross_similarity': cross_sim,
                'diversity_gap_score': diversity_gap,
                'best_match': best_match,
                'high_quality_matches': high_quality_matches,
                'topic_similarity': topic_sim,
                'topic_overlap': topic_overlap,
                'distance_similarity': distance_sim,
                'volume_similarity': volume_score,
                'cross_dataset_similarity': cross_dataset_sim
            }
        
        ranking_results[model_key] = model_scores
        
        # Print ranking for this model
        sorted_variants = sorted(variants, key=lambda x: model_scores[x]['composite_score'], reverse=True)
        print(f"\n{embeddings_dict[model_key]['model_name']} Rankings:")
        for i, variant in enumerate(sorted_variants, 1):
            score = model_scores[variant]['composite_score']
            print(f"  {i}. {variant.upper()}: {score:.4f}")
            print(f"     - Cross-similarity: {model_scores[variant]['cross_similarity']:.4f}")
            print(f"     - Best match quality: {model_scores[variant]['best_match']:.4f}")
            print(f"     - High-quality matches: {model_scores[variant]['high_quality_matches']:.3f}")
    
    # Calculate overall ranking across all models
    overall_scores = {variant: 0 for variant in variants}
    for model_scores in ranking_results.values():
        for variant in variants:
            overall_scores[variant] += model_scores[variant]['composite_score']
    
    # Average scores
    for variant in variants:
        overall_scores[variant] /= len(ranking_results)
    
    overall_ranking = sorted(variants, key=lambda x: overall_scores[x], reverse=True)
    
    print(f"\n🥇 OVERALL RANKING (Average across all models):")
    print("-" * 40)
    for i, variant in enumerate(overall_ranking, 1):
        score = overall_scores[variant]
        print(f"{i}. {variant.upper()}: {score:.4f}")
    
    return ranking_results, overall_ranking, overall_scores

# Run ranking analysis
ranking_results, overall_ranking, overall_scores = rank_synthetic_variants()

# =============================================================================
# 7. COMPREHENSIVE RESULTS COMPILATION
# =============================================================================

def compile_comprehensive_results():
    """Compile all analysis results"""
    comprehensive_results = {
        'metadata': {
            'real_samples': len(real_texts),
            'synthetic_variants': {variant: len(texts) for variant, texts in synthetic_texts.items()},
            'embedding_models': [embeddings_dict[k]['model_name'] for k in embeddings_dict.keys()],
            'analysis_types': ['semantic_similarity', 'topic_modeling', 'diversity_analysis', 'ranking']
        },
        'semantic_similarity_analysis': sts_results,
        'topic_modeling_analysis': topic_results,
        'diversity_analysis': diversity_results,
        'ranking_analysis': {
            'detailed_rankings': ranking_results,
            'overall_ranking': overall_ranking,
            'overall_scores': overall_scores
        }
    }
    
    # Save results
    results_file = EMBEDDING_DIR / 'multi_dataset_semantic_analysis_results.json'
    with open(results_file, 'w') as f:
        json.dump(comprehensive_results, f, indent=2)
    
    print(f"\n💾 Results saved to: {results_file}")
    return comprehensive_results

# Compile and save results
final_results = compile_comprehensive_results()

# =============================================================================
# 8. DETAILED SUMMARY AND RECOMMENDATIONS
# =============================================================================

print(f"\n{'='*80}")
print("🎯 MULTI-DATASET SEMANTIC ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\n📊 WINNER ANALYSIS:")
print("-" * 40)

best_variant = overall_ranking[0]
best_score = overall_scores[best_variant]

print(f"🥇 BEST PERFORMING VARIANT: {best_variant.upper()}")
print(f"   Overall Score: {best_score:.4f}")

print(f"\n📈 DETAILED COMPARISON:")
print("-" * 40)

for variant in overall_ranking:
    score = overall_scores[variant]
    print(f"\n{variant.upper()} (Score: {score:.4f}):")
    
    # Average metrics across models
    avg_cross_sim = np.mean([sts_results[model][variant]['cross_similarity']['mean'] 
                            for model in sts_results.keys()])
    avg_best_match = np.mean([sts_results[model][variant]['best_matches']['mean_best_match'] 
                             for model in sts_results.keys()])
    avg_high_quality = np.mean([sts_results[model][variant]['best_matches']['high_quality_matches'] 
                               for model in sts_results.keys()])
    
    topic_wasserstein = topic_results[variant]['topic_distribution_similarity']['wasserstein_distance']
    topic_overlap = topic_results[variant]['topic_distribution_similarity']['topic_overlap']
    
    print(f"  ✓ Cross-similarity: {avg_cross_sim:.4f}")
    print(f"  ✓ Best match quality: {avg_best_match:.4f}")
    print(f"  ✓ High-quality matches: {avg_high_quality:.3f}")
    print(f"  ✓ Topic Wasserstein: {topic_wasserstein:.4f}")
    print(f"  ✓ Topic overlap: {topic_overlap:.4f}")

print(f"\n🎨 RECOMMENDATIONS:")
print("-" * 40)
print(f"1. Use '{best_variant.upper()}' variant for best semantic similarity to real data")
print(f"2. {best_variant.upper()} shows highest cross-similarity and best match quality")
print(f"3. Consider the specific strengths of each variant for different use cases")

if best_variant == 'strong':
    print(f"4. Strong rewriting maintains semantic content while adding variation")
elif best_variant == 'weak':
    print(f"4. Weak rewriting preserves more original structure and phrasing")
else:
    print(f"4. Original rewriting provides balanced approach to synthetic generation")

print(f"\n💾 All results saved to: {EMBEDDING_DIR}/multi_dataset_semantic_analysis_results.json")
print(f"📁 Embeddings saved to: {EMBEDDING_DIR}/")
print(f"\n🎯 Analysis complete! Use the best variant ({best_variant.upper()}) for your ML training.")